In [7]:
import pandas as pd
import ast
import numpy as np


In [4]:
track_genres = pd.read_csv('./id_genres_mmsr.tsv', sep='\t')

In [5]:
# we select 1000k randomly sampled tracks to create the pairs
track_genres = track_genres.sample(n=1000, random_state=42).reset_index(drop=True)

In [6]:
track_genres['genre'] = track_genres['genre'].apply(lambda x: ast.literal_eval(str(x)))

In [10]:
samples = set([(el1, el2) for el1 in track_genres.index for el2 in track_genres.index if el1 < el2])

In [11]:
len(samples)

499500

In [12]:
10**3 * (10**3 - 1) / 2

499500.0

In [13]:
i_s = [i for i, j in samples]
j_s = [j for i, j in samples]

In [14]:
i_track_genres = track_genres.loc[i_s]
i_track_genres.columns = ['i', 'i_genres']
i_track_genres = i_track_genres.reset_index(drop=True)
j_track_genres = track_genres.loc[j_s]
j_track_genres.columns = ['j', 'j_genres']
j_track_genres = j_track_genres.reset_index(drop=True)

In [15]:
i_track_genres.head(1)

,i,i_genres
0,fIs82n32GIzRGfrQ,"[power metal, metal, progressive metal, melodi..."


In [16]:
dataset = i_track_genres.merge(j_track_genres, left_index=True, right_index=True)[['i', 'j', 'i_genres', 'j_genres']]

In [17]:
dataset.head(1)

,i,j,i_genres,j_genres
0,fIs82n32GIzRGfrQ,LSbtaNWBHzNKHa2D,"[power metal, metal, progressive metal, melodi...","[doom metal, stoner rock, psychedelic rock, me..."


In [18]:
dataset['is_match'] = dataset.apply(lambda row: len(set(row['i_genres']).intersection(set(row['j_genres']))) > 0, axis=1)

In [21]:
dataset = dataset[['i', 'j', 'is_match', 'i_genres', 'j_genres']]

In [22]:
dataset.head(10)

,i,j,is_match,i_genres,j_genres
0,fIs82n32GIzRGfrQ,LSbtaNWBHzNKHa2D,True,"[power metal, metal, progressive metal, melodi...","[doom metal, stoner rock, psychedelic rock, me..."
1,CcR8OC6ncESxbL7I,X1hvP89yKi3fqnkx,False,"[ska, reggae, lounge]","[progressive rock, rock, art rock, metal, prog..."
2,BRPqAfdKjlsfCru3,lUKXMRQbu10DU3QN,False,"[pop, dubstep, dance pop, latin, synthpop, ele...","[punk, rock, alternative rock, hard rock, hard..."
3,J2BC90nYa1vXXA3Z,8xFD1UO8nr1qwOg6,True,"[power metal, metal, melodic metal, melodic po...","[soul, pop, singer songwriter, soft rock, rock..."
4,6NNmYh1zeixjYgqx,iPwlfM8JgRDQK6Zm,True,"[freestyle, pop, miami bass, disco, electro, n...","[indie rock, rock, garage rock, pop, alternati..."
5,CfCJl3HWjZCvaTv3,Akdz6ekUC7TN3Vgd,False,"[hip hop, rap, r b, pop, funk, rock, soul, reg...","[lo fi, minimal synth]"
6,EOpAZnYeQJLvT5EU,kOVK1Bh9ihrB0CkJ,False,[technical death metal],"[soundtrack, pop, choral]"
7,Aaruqax7ZuG8p3sc,8OBTXMtAPDhZFzWY,True,"[pop, disco, new wave, synthpop, eurodance, eu...","[industrial, industrial rock, rock, alternativ..."
8,nP4fQq4Ktl5UQY6e,PCLhQBWrPpejEoT0,True,"[black metal, metal, progressive black metal]","[progressive metal, post metal, metal, metalco..."
9,Zud7kokd9rnUt7Kq,B6Ds0jDbwELjlbRN,False,"[grunge, alternative rock, rock, hard rock, al...","[house, pop, electro]"


In [23]:
# shuffle
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)
# select train val test
n = len(dataset)
train = dataset.iloc[:int(0.8*n)]
val = dataset.iloc[int(0.8*n):int(0.9*n)]
test = dataset.iloc[int(0.9*n):]

In [25]:
train.to_csv('./binary_train.csv', index=False)
val.to_csv('./binary_val.csv', index=False)
test.to_csv('./binary_test.csv', index=False)